# Verification

Is Model A's improvement over the baseline statistically significant and robust? And does Model B (+ reviews) add anything beyond Model A?

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from pathlib import Path

RESULTS_DIR = Path('results')
DATA_DIR    = Path('data')

TS_COLS  = ['ts_mean', 'ts_slope', 'ts_curvature', 'ts_range', 'ts_peak_pos']
REV_COLS = ['rev_count', 'rev_mean_rating', 'rev_positive_frac']
TARGET   = 'actual_peak_value'
ALPHAS   = [0.01, 0.1, 1.0, 10.0, 100.0]
N_PERM   = 1000
N_BOOT   = 500

def mae(actual, predicted):
    return float(np.mean(np.abs(np.array(actual, dtype=float) - np.array(predicted, dtype=float))))

def make_pipe(num_cols):
    ct = ColumnTransformer([
        ('ohe', OneHotEncoder(handle_unknown='ignore'), ['style']),
        ('sc',  StandardScaler(), num_cols),
    ])
    return Pipeline([('ct', ct), ('ridge', RidgeCV(alphas=ALPHAS))])

rng = np.random.default_rng(42)

## Verification strategy

Three questions:

1. Is the Model A vs Baseline difference significant? - Permutation test
2. Is it stable? - Bootstrap CI on the improvement
3. How much early data is needed? - Sensitivity by window length

All tests are non-parametric - they make no assumptions about normality. The permutation test is the gold standard for small samples.

In [2]:
predictions  = pd.read_csv(RESULTS_DIR / 'peak_predictions.csv')
actual    = predictions['actual'].values.astype(float)
pred_a    = predictions['pred_a'].values.astype(float)
pred_base = predictions['pred_base'].values.astype(float)

mae_a    = mae(actual, pred_a)
mae_base = mae(actual, pred_base)
diff     = mae_base - mae_a

print(f'Baseline MAE : {mae_base:.2f} pts')
print(f'Model A  MAE : {mae_a:.2f} pts')
print(f'Improvement  : {diff:+.2f} pts  ({"better" if diff > 0 else "worse"})')

Baseline MAE : 7.68 pts
Model A  MAE : 6.25 pts
Improvement  : +1.43 pts  (better)
